# **Data Cleaning**

## Objectives

The objectives of this notebook are to:

- Load the merged dataset created during the data collection stage.
- Assess the quality of the dataset by checking for missing values, duplicate records and inconsistent data types.
- Clean and standardise the dataset where necessary.
- Save the cleaned dataset for exploratory data analysis and machine learning.


## Inputs

- `data/processed/all_players.csv` – The merged Premier League player statistics dataset created in the Data Collection notebook.

## Outputs

- `data/processed/all_players_cleaned.csv` – The cleaned and validated dataset, ready for exploratory data analysis and machine learning.

## Additional Comments

This notebook focuses on data quality rather than data analysis. Cleaning tasks are performed to improve the reliability and consistency of the dataset before exploratory data analysis and model development. Any assumptions or cleaning decisions made during this stage will be documented throughout the notebook.

---

### Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\code\\premier-league-predictor\\premier-league-predictor\\jupyter_notebooks'

In [2]:
os.chdir(r"C:\code\premier-league-predictor\premier-league-predictor")
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'C:\\code\\premier-league-predictor\\premier-league-predictor'

### Import packages

All data loading and inspection in this notebook uses `pandas`. No additional libraries are needed at this stage.

In [4]:

import pandas as pd

## Load processed dataset

The merged dataset created during the data collection stage is loaded from the `data/processed` directory. This dataset will be assessed for data quality issues before exploratory data analysis.

In [5]:
data_path = "data/processed"
file_path = os.path.join(data_path, "all_players.csv")
df = pd.read_csv(file_path)

## Inspect dataset

The dataset is inspected to confirm it has loaded correctly and to obtain an initial understanding of its size, structure and data types before cleaning begins.

In [6]:
df.shape


(8206, 54)

In [7]:
df.head()

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77%,NaN,0.0,6.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-16
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78%,NaN,10.0,32.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-16
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83%,0.0,1.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-16
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-16
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-16


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8206 entries, 0 to 8205
Data columns (total 54 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    8206 non-null   object 
 1   Position                8206 non-null   object 
 2   Appearances             8206 non-null   int64  
 3   Clean sheets            3579 non-null   float64
 4   Goals conceded          3579 non-null   float64
 5   Tackles                 7244 non-null   float64
 6   Tackle success %        5531 non-null   object 
 7   Last man tackles        2617 non-null   float64
 8   Blocked shots           7244 non-null   float64
 9   Interceptions           7244 non-null   float64
 10  Clearances              7244 non-null   float64
 11  Headed Clearance        7244 non-null   float64
 12  Clearances off line     2617 non-null   float64
 13  Recoveries              5531 non-null   float64
 14  Duels won               5531 non-null   

## Investigate missing values

Missing values are assessed to determine whether they represent genuine missing data or values that are not applicable for certain player positions. Understanding the reason for missing values is essential before deciding how they should be handled.

In [9]:
df.isnull().sum().sort_values(ascending=False)

Throw outs                7244
Sweeper clearances        7244
Catches                   7244
High Claims               7244
Punches                   7244
Penalties saved           7244
Saves                     7244
Goal Kicks                7244
Clearances off line       5589
Last man tackles          5589
Goals conceded            4627
Own goals                 4627
Clean sheets              4627
Shots                     3579
Penalties scored          3579
Big chances missed        3579
Shots on target           3579
Shooting accuracy %       3579
Goals per match           3579
Freekicks scored          3579
Aerial battles won        2675
Recoveries                2675
Cross accuracy %          2675
Duels won                 2675
Successful 50/50s         2675
Duels lost                2675
Tackle success %          2675
Through balls             2675
Aerial battles lost       2675
Accurate long balls       1713
Errors leading to goal    1713
Tackles                    962
Offsides

### Investigate goalkeeper-specific statistics

Several goalkeeper-specific features contain a large number of missing values. This investigation determines whether these missing values represent poor data quality or statistics that are only applicable to goalkeepers.

In [10]:
df["Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Goalkeeper     962
Name: count, dtype: int64

In [11]:
goalkeepers = (df["Position"] == "Goalkeeper").sum()

In [12]:
outfield_players = len(df) - goalkeepers

In [13]:
missing_saves = df["Saves"].isna().sum()

In [14]:
print(f"Goalkeepers: {goalkeepers}")
print(f"Outfield players: {outfield_players}")
print(f"Missing 'Saves' values: {missing_saves}")

Goalkeepers: 962
Outfield players: 7244
Missing 'Saves' values: 7244


In [15]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
]

df[goalkeeper_columns].isna().sum()

Saves                 7244
Penalties saved       7244
Punches               7244
High Claims           7244
Catches               7244
Sweeper clearances    7244
Throw outs            7244
Goal Kicks            7244
dtype: int64

### Findings

The investigation confirmed that all goalkeeper-specific statistics contain exactly **7,244** missing values, matching the number of outfield players in the dataset. This demonstrates that these values are not missing due to incomplete data collection; they are not applicable to outfield players. These columns will therefore be handled differently from genuinely missing data during the cleaning process.

### Investigate defensive statistics

Two defensive features, `Last man tackles` and `Clearances off line`, contain a large number of missing values. This investigation determines whether these missing values are associated with particular playing positions or represent incomplete data.

In [16]:
defenders = (df["Position"] == "Defender").sum()

In [17]:
other_players = len(df) - defenders

In [18]:
missing_last_man_tackles = df["Last man tackles"].isna().sum()

In [19]:
print(f"Defenders: {defenders}")
print(f"Other players: {other_players}")
print(f"Missing 'Last man tackles' values: {missing_last_man_tackles}")

Defenders: 2635
Other players: 5571
Missing 'Last man tackles' values: 5589


In [20]:
df.loc[df["Last man tackles"].notna(), "Position"].value_counts()

Position
Defender      2609
Midfielder       8
Name: count, dtype: int64

### Findings

The investigation showed that `Last man tackles` is primarily recorded for defenders, with only a small number of midfielders having recorded values. This indicates that the missing values are expected for most attacking players rather than representing missing or incomplete data.

### Investigate attacking statistics

Several attacking statistics also contain a large number of missing values. This investigation determines whether these missing values represent statistics that are only applicable to certain players or whether they indicate missing or incomplete data.

In [21]:
df.loc[df["Shots"].notna(), "Position"].value_counts()

Position
Midfielder    2876
Forward       1725
Defender        26
Name: count, dtype: int64

In [22]:
for position in df["Position"].unique():
    missing = df.loc[df["Position"] == position, "Shots"].isna().sum()
    print(position, missing)

Midfielder 8
Defender 2609
Forward 0
Goalkeeper 962


In [23]:
df.loc[
    (df["Position"] == "Defender") & (df["Shots"].isna()),
    ["Name", "Appearances", "Goals", "Shots"]
].head(20)

,Name,Appearances,Goals,Shots
2,Abdul Rahman Baba,15,0,NaN
12,Nathan Aké,24,1,NaN
14,Alberto Moreno,32,1,NaN
16,Toby Alderweireld,38,4,NaN
18,Trent Alexander-Arnold,0,0,NaN
25,Daniel Amartey,5,0,NaN
26,Jordan Amavi,10,0,NaN
31,Angeliño,0,0,NaN
32,Gabriele Angella,0,0,NaN
45,César Azpilicueta,37,2,NaN


In [24]:
df.loc[
    (df["Goals"] > 0) & (df["Shots"].isna()),
    ["Name", "Position", "Goals", "Shots"]
]

,Name,Position,Goals,Shots
12,Nathan Aké,Defender,1,NaN
14,Alberto Moreno,Defender,1,NaN
16,Toby Alderweireld,Defender,4,NaN
45,César Azpilicueta,Defender,2,NaN
48,Leighton Baines,Defender,2,NaN
...,...,...,...,...
8168,Ben White,Defender,4,NaN
8192,Illia Zabarnyi,Defender,1,NaN
8196,Zanka,Defender,1,NaN
8201,Oleksandr Zinchenko,Defender,1,NaN


### Findings

Unlike the goalkeeper-specific statistics, the missing values in attacking statistics do not appear to be structural. The investigation identified **635 player records** with at least one recorded goal but missing shot statistics. Since a player cannot score without taking a shot, this indicates that these missing values represent incomplete source data rather than statistics that are not applicable.

***

## Decide how to handle missing values

The previous investigations identified different types of missing values within the dataset. Rather than applying a single approach to all missing data, each group of features will be handled according to the reason the values are missing.

### Goalkeeper-specific statistics

The investigation confirmed that missing values in goalkeeper-specific statistics are structural rather than the result of incomplete data. As these statistics are not applicable to outfield players, the missing values are replaced with `0` to indicate that no value was recorded for those statistics.

In [25]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
]

In [26]:
df[goalkeeper_columns] = df[goalkeeper_columns].fillna(0)
df[goalkeeper_columns].isna().sum()

Saves                 0
Penalties saved       0
Punches               0
High Claims           0
Catches               0
Sweeper clearances    0
Throw outs            0
Goal Kicks            0
dtype: int64

### Outcome

The goalkeeper-specific features no longer contain missing values. These `NaN` values were replaced with `0` because the statistics are not applicable to outfield players rather than representing incomplete data.

In [27]:
df.columns.tolist()

['Name',
 'Position',
 'Appearances',
 'Clean sheets',
 'Goals conceded',
 'Tackles',
 'Tackle success %',
 'Last man tackles',
 'Blocked shots',
 'Interceptions',
 'Clearances',
 'Headed Clearance',
 'Clearances off line',
 'Recoveries',
 'Duels won',
 'Duels lost',
 'Successful 50/50s',
 'Aerial battles won',
 'Aerial battles lost',
 'Own goals',
 'Errors leading to goal',
 'Assists',
 'Passes',
 'Passes per match',
 'Big chances created',
 'Crosses',
 'Cross accuracy %',
 'Through balls',
 'Accurate long balls',
 'Yellow cards',
 'Red cards',
 'Fouls',
 'Offsides',
 'Goals',
 'Headed goals',
 'Goals with right foot',
 'Goals with left foot',
 'Hit woodwork',
 'Goals per match',
 'Penalties scored',
 'Freekicks scored',
 'Shots',
 'Shots on target',
 'Shooting accuracy %',
 'Big chances missed',
 'Saves',
 'Penalties saved',
 'Punches',
 'High Claims',
 'Catches',
 'Sweeper clearances',
 'Throw outs',
 'Goal Kicks',
 'Season']

In [28]:
attacking_columns = [
    "Goals",
    "Goals per match",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
    "Hit woodwork",
    "Assists",
    "Big chances created",
]

In [29]:
df[attacking_columns].isna().sum()

Goals                     0
Goals per match        3579
Shots                  3579
Shots on target        3579
Shooting accuracy %    3579
Big chances missed     3579
Hit woodwork            962
Assists                   0
Big chances created     962
dtype: int64

In [30]:
df.loc[df["Hit woodwork"].notna(), "Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Name: count, dtype: int64

In [31]:
df.loc[df["Big chances created"].notna(), "Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Name: count, dtype: int64

## Summary of missing value categories

| Category | Features | Planned action |
|----------|----------|----------------|
| Structural missing values | Saves, Penalties saved, Punches, High Claims, Catches, Sweeper clearances, Throw outs, Goal Kicks, Hit woodwork, Big chances created | Replace `NaN` with `0` |
| Position-specific statistics | Last man tackles, Clearances off line | Investigate further before deciding |
| Incomplete source data | Goals per match, Shots, Shots on target, Shooting accuracy %, Big chances missed | Do not replace with `0`; investigate appropriate handling |

### Update structural missing value features

Further investigation showed that `Hit woodwork` and `Big chances created` are complete for all outfield players and only missing for goalkeepers. These features are therefore added to the list of structural missing values and their missing values are replaced with `0`.

In [32]:
structural_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
    "Hit woodwork",
    "Big chances created",
]

df[structural_columns] = df[structural_columns].fillna(0)

In [33]:
df[structural_columns].isna().sum()

Saves                  0
Penalties saved        0
Punches                0
High Claims            0
Catches                0
Sweeper clearances     0
Throw outs             0
Goal Kicks             0
Hit woodwork           0
Big chances created    0
dtype: int64

### Outcome

The structural missing values have now been replaced with `0`. A verification check confirmed that these features no longer contain missing values. This approach was appropriate because the missing values represented statistics that were not applicable to certain player positions rather than incomplete data.

***

## Handle position-specific statistics

The investigation showed that some statistics are primarily associated with specific playing positions rather than being applicable to all players. The following section determines the most appropriate way to handle these remaining missing values.

In [34]:
position_specific_columns = [
    "Last man tackles",
    "Clearances off line",
]

In [35]:
df[position_specific_columns].isna().sum()

Last man tackles       5589
Clearances off line    5589
dtype: int64

### Investigation findings

The investigation showed that `Last man tackles` and `Clearances off line` contain the same number of missing values. These statistics are primarily recorded for defenders, with only a very small number of midfielders having recorded values. The missing values are therefore position-specific rather than random.

***

## Handle incomplete source data

The remaining missing values occur in attacking statistics where the investigation showed that the data is incomplete rather than structurally missing. These values are reviewed separately to determine whether they should be retained, removed or imputed.

In [36]:
incomplete_columns = [
    "Goals per match",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
]

In [37]:
df[incomplete_columns].isna().sum()

Goals per match        3579
Shots                  3579
Shots on target        3579
Shooting accuracy %    3579
Big chances missed     3579
dtype: int64

### Investigation findings

The investigation showed that these attacking statistics contain genuine missing values rather than structural missing values. Players with recorded goals were found to have missing shot statistics, demonstrating that the source data is incomplete. Replacing these values with `0` would introduce inaccurate information into the dataset.

In [38]:
df.isna().sum().sort_values(ascending=False)

Clearances off line       5589
Last man tackles          5589
Goals conceded            4627
Own goals                 4627
Clean sheets              4627
Shots                     3579
Shots on target           3579
Goals per match           3579
Freekicks scored          3579
Penalties scored          3579
Big chances missed        3579
Shooting accuracy %       3579
Aerial battles won        2675
Cross accuracy %          2675
Through balls             2675
Aerial battles lost       2675
Duels won                 2675
Tackle success %          2675
Duels lost                2675
Successful 50/50s         2675
Recoveries                2675
Errors leading to goal    1713
Accurate long balls       1713
Tackles                    962
Headed goals               962
Crosses                    962
Interceptions              962
Blocked shots              962
Headed Clearance           962
Clearances                 962
Goals with left foot       962
Goals with right foot      962
Offsides

In [39]:
df.loc[df["Clean sheets"].notna(), "Position"].value_counts()

Position
Defender      2609
Goalkeeper     962
Midfielder       8
Name: count, dtype: int64

In [40]:
df.loc[df["Goals conceded"].notna(), "Position"].value_counts()

Position
Defender      2609
Goalkeeper     962
Midfielder       8
Name: count, dtype: int64

In [41]:
df.loc[
    (df["Position"] == "Defender") &
    (df["Clean sheets"].isna()),
    ["Name", "Appearances", "Goals", "Clean sheets"]
]

,Name,Appearances,Goals,Clean sheets
463,Sam McQueen,0,0,NaN
533,Paddy McCarthy,0,0,NaN
1025,Dael Fry,0,0,NaN
1280,Sam McQueen,13,0,NaN
1366,Paddy McCarthy,0,0,NaN
1592,Oleksandr Zinchenko,0,0,NaN
2080,Sam McQueen,7,0,NaN
2387,Oleksandr Zinchenko,8,0,NaN
2528,Trevoh Chalobah,0,0,NaN
2718,Kortney Hause,0,0,NaN


### Further investigation

Further investigation of defensive statistics such as `Clean sheets` identified players with substantial numbers of appearances but missing values. This suggests that these missing values also represent incomplete source data rather than statistics that are not applicable.

### Investigate position-dependent performance statistics

Several performance statistics contain **2,675** missing values. The following investigation determines whether these missing values are position-specific or represent incomplete source data.

In [42]:
df.loc[df["Duels won"].notna(), "Position"].value_counts()

Position
Midfielder    2853
Defender      2635
Forward         43
Name: count, dtype: int64

In [43]:
df.loc[
    (df["Position"] == "Forward") &
    (df["Duels won"].isna()),
    ["Name", "Appearances", "Goals", "Duels won", "Recoveries"]
].head(20)

,Name,Appearances,Goals,Duels won,Recoveries
4,Tammy Abraham,2,0,NaN,NaN
6,Emmanuel Adebayor,12,1,NaN,NaN
9,Benik Afobe,15,4,NaN,NaN
10,Gabriel Agbonlahor,15,1,NaN,NaN
11,Sergio Agüero,30,24,NaN,NaN
13,Chuba Akpom,0,0,NaN,NaN
19,Alexandre Pato,2,1,NaN,NaN
33,Victor Anichebe,10,0,NaN,NaN
38,Adam Armstrong,0,0,NaN,NaN
39,Marko Arnautovic,34,11,NaN,NaN


### Decision

The remaining missing values will be retained as `NaN`. The investigations showed that these values do not represent structural missing values and cannot be safely replaced with `0` without introducing inaccurate information. The remaining missing values represent either incomplete source data or statistics that were not consistently recorded across all player roles. Any further handling of these values will be performed during feature selection and model preparation, depending on the requirements of the chosen machine learning model.

***

## Check for duplicate records

Duplicate records can introduce bias into exploratory analysis and machine learning models by giving additional weight to repeated observations. The dataset is checked for duplicate rows before further cleaning.

In [44]:
df.duplicated().sum()

np.int64(6)

### Investigate duplicate records

The dataset contains duplicate records. Before removing them, the duplicate rows are inspected to determine whether they are true duplicates or repeated observations that should be retained.

In [45]:
duplicates = df[df.duplicated()]

duplicates.head(6)

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
6139,Joseph Anang,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6140,Joachim Andersen,Defender,32,8.0,37.0,42.0,52%,0.0,2.0,24.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6143,André Gomes,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6144,Andreas Pereira,Midfielder,33,NaN,NaN,18.0,50%,NaN,14.0,6.0,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6145,Andrey Santos,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6147,Tino Anjorin,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23


In [46]:
duplicates.nunique()

Name                      6
Position                  3
Appearances               3
Clean sheets              2
Goals conceded            2
Tackles                   3
Tackle success %          3
Last man tackles          1
Blocked shots             3
Interceptions             3
Clearances                3
Headed Clearance          3
Clearances off line       1
Recoveries                3
Duels won                 3
Duels lost                3
Successful 50/50s         3
Aerial battles won        3
Aerial battles lost       3
Own goals                 2
Errors leading to goal    2
Assists                   2
Passes                    3
Passes per match          3
Big chances created       3
Crosses                   3
Cross accuracy %          3
Through balls             3
Accurate long balls       3
Yellow cards              2
Red cards                 1
Fouls                     3
Offsides                  2
Goals                     3
Headed goals              2
Goals with right foo

In [47]:
df[df.duplicated(keep=False)].sort_values("Name")

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
6103,Andreas Pereira,Midfielder,33,NaN,NaN,18.0,50%,NaN,14.0,6.0,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6144,Andreas Pereira,Midfielder,33,NaN,NaN,18.0,50%,NaN,14.0,6.0,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6104,Andrey Santos,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6145,Andrey Santos,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6102,André Gomes,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6143,André Gomes,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6100,Joachim Andersen,Defender,32,8.0,37.0,42.0,52%,0.0,2.0,24.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6140,Joachim Andersen,Defender,32,8.0,37.0,42.0,52%,0.0,2.0,24.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6099,Joseph Anang,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6139,Joseph Anang,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23


### Investigation findings

The initial duplicate check identified **1,313** duplicate records. Further investigation showed that the merged dataset at that stage did not contain a `Season` column, meaning players with identical statistics across different seasons became indistinguishable.

The Data Collection notebook was therefore updated to preserve season information before repeating the duplicate investigation.

### Investigate remaining duplicate records

After adding the `Season` identifier, the number of duplicate records reduced from 1,313 to 6. These remaining duplicates are inspected to determine whether they are genuine duplicate observations or valid records that should be retained.

In [48]:
duplicates = df[df.duplicated(keep=False)]

duplicates

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
6099,Joseph Anang,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6100,Joachim Andersen,Defender,32,8.0,37.0,42.0,52%,0.0,2.0,24.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6102,André Gomes,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6103,Andreas Pereira,Midfielder,33,NaN,NaN,18.0,50%,NaN,14.0,6.0,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6104,Andrey Santos,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6107,Tino Anjorin,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6139,Joseph Anang,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6140,Joachim Andersen,Defender,32,8.0,37.0,42.0,52%,0.0,2.0,24.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6143,André Gomes,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23
6144,Andreas Pereira,Midfielder,33,NaN,NaN,18.0,50%,NaN,14.0,6.0,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-23


In [49]:
duplicates[["Name", "Season", "Position", "Appearances", "Goals"]]

,Name,Season,Position,Appearances,Goals
6099,Joseph Anang,2022-23,Goalkeeper,0,0
6100,Joachim Andersen,2022-23,Defender,32,1
6102,André Gomes,2022-23,Midfielder,0,0
6103,Andreas Pereira,2022-23,Midfielder,33,4
6104,Andrey Santos,2022-23,Midfielder,0,0
6107,Tino Anjorin,2022-23,Midfielder,0,0
6139,Joseph Anang,2022-23,Goalkeeper,0,0
6140,Joachim Andersen,2022-23,Defender,32,1
6143,André Gomes,2022-23,Midfielder,0,0
6144,Andreas Pereira,2022-23,Midfielder,33,4


### Investigation findings

Following the addition of the `Season` column, only six duplicate records remained. Inspection confirmed that these records were genuine duplicates within the same season rather than observations from different seasons. As they provided no additional information, the duplicate rows were removed from the dataset.

In [50]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

### Result

The duplicate records were successfully removed, leaving a dataset containing only unique player-season observations.

***

## Validate data types

Correct data types are important for data analysis and machine learning. This section checks that each column has an appropriate data type and identifies any columns that require conversion before further analysis.

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8200 entries, 0 to 8205
Data columns (total 54 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    8200 non-null   object 
 1   Position                8200 non-null   object 
 2   Appearances             8200 non-null   int64  
 3   Clean sheets            3577 non-null   float64
 4   Goals conceded          3577 non-null   float64
 5   Tackles                 7239 non-null   float64
 6   Tackle success %        5526 non-null   object 
 7   Last man tackles        2616 non-null   float64
 8   Blocked shots           7239 non-null   float64
 9   Interceptions           7239 non-null   float64
 10  Clearances              7239 non-null   float64
 11  Headed Clearance        7239 non-null   float64
 12  Clearances off line     2616 non-null   float64
 13  Recoveries              5526 non-null   float64
 14  Duels won               5526 non-null   float

### Investigate percentage columns

Most variables already have appropriate data types. However, three percentage-based features are stored as text (`object`) rather than numeric values. These columns are investigated before deciding whether they should be converted to numeric data types.

In [52]:
df[
    [
        "Tackle success %",
        "Cross accuracy %",
        "Shooting accuracy %"
    ]
].head(20)

,Tackle success %,Cross accuracy %,Shooting accuracy %
0,77%,25%,50%
1,78%,31%,26%
2,83%,16%,NaN
3,0%,0%,0%
4,NaN,NaN,0%
5,78%,30%,14%
6,NaN,NaN,33%
7,NaN,NaN,NaN
8,81%,16%,21%
9,NaN,NaN,36%


### Convert percentage columns

The percentage-based features are currently stored as text because they include the `%` symbol. The symbol is removed and the columns are converted to numeric values so they can be used in statistical analysis and machine learning models.

In [53]:
percentage_columns = [
    "Tackle success %",
    "Cross accuracy %",
    "Shooting accuracy %"
]

for column in percentage_columns:
    df[column] = (
        df[column]
        .str.replace("%", "", regex=False)
        .astype(float)
    )

In [54]:
df[percentage_columns].dtypes

Tackle success %       float64
Cross accuracy %       float64
Shooting accuracy %    float64
dtype: object

### Outcome

The percentage-based features were successfully converted from text to numeric values by removing the `%` symbol and converting the cleaned values to `float`. These columns are now suitable for numerical analysis and machine learning.

***

### Investigate `Passes` data type

The `Passes` feature is stored as text despite representing a numeric count. The values are inspected to identify any formatting that prevents pandas from recognising the column as numeric.

In [56]:
df["Passes"].head(20)

0       119
1       938
2       526
3         0
4        10
5       546
6       230
7       853
8     1,274
9       224
10      223
11      727
12      609
13        0
14    1,323
15      912
16    2,076
17        0
18        0
19       27
Name: Passes, dtype: object

In [57]:
df["Passes"].unique()[:20]

array(['119', '938', '526', '0', '10', '546', '230', '853', '1,274',
       '224', '223', '727', '609', '1,323', '912', '2,076', '27', '410',
       '1,083', '8'], dtype=object)

### Convert `Passes` to numeric

The `Passes` column is stored as text because some values include commas as thousands separators. These commas are removed and the column is converted to a numeric data type so it can be used in analysis and machine learning.

In [58]:
df["Passes"] = (
    df["Passes"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [59]:
df["Passes"].dtype

dtype('int64')

In [60]:
df["Passes"].head(10)

0     119
1     938
2     526
3       0
4      10
5     546
6     230
7     853
8    1274
9     224
Name: Passes, dtype: int64

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8200 entries, 0 to 8205
Data columns (total 54 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    8200 non-null   object 
 1   Position                8200 non-null   object 
 2   Appearances             8200 non-null   int64  
 3   Clean sheets            3577 non-null   float64
 4   Goals conceded          3577 non-null   float64
 5   Tackles                 7239 non-null   float64
 6   Tackle success %        5526 non-null   float64
 7   Last man tackles        2616 non-null   float64
 8   Blocked shots           7239 non-null   float64
 9   Interceptions           7239 non-null   float64
 10  Clearances              7239 non-null   float64
 11  Headed Clearance        7239 non-null   float64
 12  Clearances off line     2616 non-null   float64
 13  Recoveries              5526 non-null   float64
 14  Duels won               5526 non-null   float

### Outcome

The data type review confirmed that the main text fields (`Name`, `Position` and `Season`) are correctly stored as `object` values. The `Passes` column was stored as text because some values contained commas as thousands separators. The commas were removed and the column was converted to an integer data type so it could be used in numerical analysis and machine learning

***

## Validate data values

The dataset is checked for impossible or unrealistic values that may indicate data quality issues. This includes verifying that statistics which cannot logically be negative do not contain invalid values.

In [62]:
df.describe()

,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,Clearances,Headed Clearance,...,Shooting accuracy %,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks
count,8200.000000,3577.000000,3577.000000,7239.000000,5526.000000,2616.000000,7239.000000,7239.000000,7239.000000,7239.00000,...,4623.000000,4623.000000,8200.000000,8200.000000,8200.000000,8200.000000,8200.000000,8200.000000,8200.000000,8200.000000
mean,10.993902,2.259435,12.599105,15.074872,33.125588,0.149465,3.120459,9.949164,18.376019,9.58392,...,16.898983,1.327709,2.365244,0.015610,0.337317,0.616341,0.180244,0.404878,3.448537,5.914146
std,13.417361,3.752823,17.605936,22.618145,33.615631,0.470368,5.749430,16.670578,37.181936,20.40328,...,21.050165,3.000740,14.507229,0.153823,2.300590,4.132030,1.282587,2.786281,21.583305,36.055864
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,2.000000,39.000000,0.000000,0.000000,1.000000,1.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,23.000000,4.000000,24.000000,24.000000,63.000000,0.000000,4.000000,14.000000,19.000000,9.00000,...,33.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,124.000000,21.000000,89.000000,181.000000,100.000000,4.000000,48.000000,156.000000,352.000000,237.00000,...,100.000000,34.000000,166.000000,3.000000,66.000000,81.000000,34.000000,46.000000,286.000000,378.000000


In [64]:
numeric_columns = df.select_dtypes(include=["number"]).columns

for column in numeric_columns:
    if (df[column] < 0).any():
        print(f"{column}: Negative values found")

### Investigation findings

All numeric features were checked for negative values. No negative values were identified, indicating that the dataset does not contain any impossible values for statistics that represent counts or measurements.

### Validate percentage values

The percentage-based features are checked to ensure that all values fall within the valid range of 0 to 100 percent.

In [65]:
percentage_columns = [
    "Tackle success %",
    "Cross accuracy %",
    "Shooting accuracy %"
]

for column in percentage_columns:
    invalid = df[(df[column] < 0) | (df[column] > 100)]
    print(column, len(invalid))

Tackle success % 0
Cross accuracy % 0
Shooting accuracy % 0


### Validate Player Appearances

During exploratory data analysis, four player-season records were identified with more than 38 appearances. As a Premier League club plays a maximum of 38 league matches in a season, these records are inconsistent with the season-level structure of the dataset.

Further inspection suggested that other statistics within these records may also contain cumulative rather than single-season values. Therefore, the affected records are removed rather than individual values being manually corrected.

In [66]:
df[df["Appearances"] > 38][
    ["Name", "Position", "Appearances", "Goals", "Season"]
]

,Name,Position,Appearances,Goals,Season
894,Nacer Chadli,Midfielder,124,21,2016-17
1251,Dean Marney,Midfielder,96,4,2016-17
2482,Yannick Bolasie,Forward,119,11,2018-19
3049,Alex Pritchard,Midfielder,48,3,2018-19


In [67]:
df[df["Appearances"] <= 38]


,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77.0,NaN,0.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78.0,NaN,10.0,32.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83.0,0.0,1.0,23.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8201,Oleksandr Zinchenko,Defender,27,3.0,15.0,50.0,60.0,0.0,10.0,20.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-24
8202,Hakim Ziyech,Midfielder,0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-24
8203,Kurt Zouma,Defender,33,3.0,63.0,23.0,74.0,1.0,4.0,29.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-24
8204,Oliwier Zych,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2023-24


In [68]:
df = df[df["Appearances"] <= 38]

In [69]:
df.shape

(8196, 54)

In [70]:
(df["Appearances"] > 38).sum()

np.int64(0)

**Validation:** Four invalid player-season records were removed. The dataset now contains 8,196 records, with no remaining appearance values greater than 38.

***

## Save cleaned dataset

Following the data cleaning and validation process, the cleaned dataset is saved to the `data/processed` directory. This version will be used for exploratory data analysis and machine learning in the following notebooks.

In [71]:
cleaned_players_df = df


In [72]:
output_path = "data/processed/all_players_cleaned.csv"

cleaned_players_df.to_csv(output_path, index=False)

print(f"Dataset saved to {output_path}")

Dataset saved to data/processed/all_players_cleaned.csv


## Conclusions and Next Steps

### Conclusions

The dataset has been cleaned and validated by:

- Investigating and handling missing values.
- Identifying and removing genuine duplicate records.
- Correcting data types where required.
- Validating numeric and percentage values.
- Preserving season information for each player record.

The cleaned dataset is now suitable for exploratory data analysis and subsequent feature engineering. Remaining genuine missing values have been intentionally retained for appropriate treatment during model preparation.

### Next Steps

The next notebook will focus on exploratory data analysis (EDA). This will include exploring feature distributions, identifying relationships between variables, and selecting appropriate features for predictive modelling.